# 2.2 数据清洗

## 学习目标
- 识别并处理缺失值
- 检测和处理价格异常值（如停牌、错误数据）
- 理解股票复权的必要性与方法
- 对齐不同资产的时间序列

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams['figure.figsize'] = (12, 4)

# 下载数据
raw = yf.download(['AAPL', 'MSFT', 'JPM'], start='2020-01-01',
                  end='2024-01-01', progress=False)['Close']
print(f'原始数据形状: {raw.shape}')
raw.head()

## 1. 缺失值检测与处理

In [ ]:
# 人为注入缺失值（模拟真实情况）
data = raw.copy()
np.random.seed(42)
mask_rows = np.random.choice(len(data), 20, replace=False)
mask_cols = np.random.choice(len(data.columns), 20, replace=True)
for r, c in zip(mask_rows, mask_cols):
    data.iloc[r, c] = np.nan

print('缺失值统计：')
print(data.isnull().sum())
print(f'\n总缺失率: {data.isnull().sum().sum() / data.size:.2%}')

In [ ]:
# 处理方法对比
methods = {
    '前向填充 (ffill)': data.fillna(method='ffill'),
    '后向填充 (bfill)': data.fillna(method='bfill'),
    '线性插值': data.interpolate(method='linear'),
    '删除含NaN行': data.dropna(),
}
print('各方法处理后的数据量：')
for name, df in methods.items():
    print(f'  {name:<20}: {len(df)} 行, 剩余NaN: {df.isnull().sum().sum()}')

# 推荐：金融数据常用前向填充（节假日使用上一个交易日价格）
clean_data = data.fillna(method='ffill').fillna(method='bfill')
print(f'\n最终清洗后 NaN 数量: {clean_data.isnull().sum().sum()}')

## 2. 异常值检测

In [ ]:
# 基于收益率的异常值检测
returns = clean_data.pct_change().dropna()

# 方法1：3-sigma 法则
def detect_outliers_sigma(ret_series, n_sigma=3):
    mean, std = ret_series.mean(), ret_series.std()
    lower, upper = mean - n_sigma * std, mean + n_sigma * std
    outliers = ret_series[(ret_series < lower) | (ret_series > upper)]
    return outliers, lower, upper

ticker = 'AAPL'
outliers, lower, upper = detect_outliers_sigma(returns[ticker])
print(f'{ticker} 3-sigma 异常值数量: {len(outliers)}')

fig, ax = plt.subplots()
ax.plot(returns.index, returns[ticker], linewidth=0.8, color='steelblue', alpha=0.7)
ax.scatter(outliers.index, outliers.values, color='red', s=40, zorder=5, label='异常值')
ax.axhline(upper, color='orange', linestyle='--', alpha=0.7, label=f'+3σ = {upper:.2%}')
ax.axhline(lower, color='orange', linestyle='--', alpha=0.7, label=f'-3σ = {lower:.2%}')
ax.set_title(f'{ticker} 日收益率异常值检测（3-sigma）', fontsize=13)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 股票复权

分红和拆股会使价格产生"跳变"，复权是为了保证价格序列的**连续性**。

- **前复权（qfq）**：以最新价为基准，历史价格向前调整 → 适合看图、计算收益率
- **后复权（hfq）**：以上市价为基准，向后调整 → 适合长期策略研究

In [ ]:
# 对比复权与不复权价格（以 AAPL 为例，历史有多次拆股）
aapl_adj = yf.download('AAPL', start='2000-01-01', end='2024-01-01',
                        progress=False, auto_adjust=True)['Close'].squeeze()
aapl_raw = yf.download('AAPL', start='2000-01-01', end='2024-01-01',
                        progress=False, auto_adjust=False)['Close'].squeeze()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(aapl_adj.index, aapl_adj.values, label='复权价格 (auto_adjust=True)', linewidth=1.5)
ax.plot(aapl_raw.index, aapl_raw.values, label='原始价格 (auto_adjust=False)',
        linewidth=1.5, linestyle='--', alpha=0.8)
ax.set_title('AAPL 复权 vs 原始价格（2000-2024）', fontsize=13)
ax.set_ylabel('价格 (USD)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print('提示：量化研究应始终使用复权价格计算收益率！')

## 4. 多资产时间对齐

In [ ]:
# 不同市场的交易日不完全对齐（节假日不同）
us = yf.download('SPY', start='2023-01-01', end='2023-03-01', progress=False)['Close']
cn_dates = pd.bdate_range('2023-01-01', '2023-03-01', freq='C',
                           holidays=['2023-01-02', '2023-01-23', '2023-01-24'])  # 简化

print(f'美股交易日数: {len(us)}')
print(f'A股工作日数 (估算): {len(cn_dates)}')

# 对齐：取共同的日期（outer join 后填充）
# 多资产对齐最佳实践
assets = yf.download(['AAPL', 'MSFT', 'SPY'], start='2023-01-01',
                      end='2024-01-01', progress=False)['Close']
# 删去任何资产有 NaN 的行
aligned = assets.dropna(how='any')
print(f'\n对齐前: {len(assets)} 行，对齐后: {len(aligned)} 行')

## 🎯 练习

1. 找一个含有停牌日（连续几天成交量为0）的A股数据，观察停牌期间价格如何处理。
2. 使用 IQR（四分位距）方法代替 3-sigma 检测异常值，比较两者结果差异。
3. 2020年3月新冠暴跌期间，AAPL 的极端日收益率是否被 3-sigma 标记为"异常"？

---
**下一节** → `03_feature_engineering.ipynb`